# Lab 4: Named Entity Recognition (NER) with Transformers

Goals:
1) Use a BERT-based token classification model for NER.
2) Prompt a Gemma chat model to perform NER.
3) Evaluate results for both approaches.

We'll use a small English dataset with PERSON/ORG/LOC entities.

## Setup
We adopt the same caching pattern as previous labs. Gemma generation is enabled in this notebook.

In [4]:
import os, json, re
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification, AutoModelForCausalLM, pipeline

BASE_DIR = os.path.join('lab4/')
# BASE_DIR = os.path.join('../../lab4')
DATA_DIR = os.path.join(BASE_DIR, 'data')
CACHE_DIR = os.path.join(BASE_DIR, 'models_cache')
os.makedirs(CACHE_DIR, exist_ok=True)

print('Transformers:', __import__('transformers').__version__)
print('Torch:', torch.__version__)

Transformers: 4.56.0
Torch: 2.8.0+cu128


In [2]:
import requests

def download_file(url, destination):
    response = requests.get(url)
    if response.status_code == 200:
        with open(destination, 'wb') as f:
            f.write(response.content)
        print(f"Downloaded {url} to {destination}")
    else:
        print(f"Failed to download {url}: Status code {response.status_code}")

# Ensure DATA_DIR exists
os.makedirs(DATA_DIR, exist_ok=True)

urls_to_download = [
    "https://raw.githubusercontent.com/ferrazzipietro//nlp-2026-labs-dei/main/lab4/data/ner_examples.json",
    "https://raw.githubusercontent.com/ferrazzipietro//nlp-2026-labs-dei/main/lab4/data/few_shot_ner_examples.json",
    "https://raw.githubusercontent.com/ferrazzipietro//nlp-2026-labs-dei/main/lab4/data/ner_examples_alt.json"
]
for url in urls_to_download:
  path = os.path.join(DATA_DIR, os.path.basename(url))
  download_file(url, path)

Downloaded https://raw.githubusercontent.com/ferrazzipietro//nlp-2026-labs-dei/main/lab4/data/ner_examples.json to lab4/data/ner_examples.json
Downloaded https://raw.githubusercontent.com/ferrazzipietro//nlp-2026-labs-dei/main/lab4/data/few_shot_ner_examples.json to lab4/data/few_shot_ner_examples.json
Downloaded https://raw.githubusercontent.com/ferrazzipietro//nlp-2026-labs-dei/main/lab4/data/ner_examples_alt.json to lab4/data/ner_examples_alt.json


## Part 1: BERT-based NER (token classification)


In [5]:
# Fine-tuning BERT on a classic NER corpus (WikiANN English)
from datasets import load_dataset
from transformers import DataCollatorForTokenClassification, Trainer, TrainingArguments

BERT_FT_ID = 'bert-base-cased'
NER_LABELS = ['O', 'B-PERSON', 'I-PERSON', 'B-ORG', 'I-ORG', 'B-LOCATION', 'I-LOCATION']
label2id_ft = {label: idx for idx, label in enumerate(NER_LABELS)}
id2label_ft = {idx: label for label, idx in label2id_ft.items()}

tokenizer_ft = AutoTokenizer.from_pretrained(BERT_FT_ID, cache_dir=CACHE_DIR)
model_ft = AutoModelForTokenClassification.from_pretrained(
    BERT_FT_ID,
    num_labels=len(NER_LABELS),
    id2label=id2label_ft,
    label2id=label2id_ft,
    cache_dir=CACHE_DIR,
)

raw_ds = load_dataset('wikiann', 'en')
tag_names = raw_ds['train'].features['ner_tags'].feature.names
label_name_map = {'PER': 'PERSON', 'ORG': 'ORG', 'LOC': 'LOCATION'}

# Keep the example small enough to run quickly while still using a real benchmark dataset.
train_ds = raw_ds['train'].shuffle(seed=42).select(range(256))
eval_ds = raw_ds['validation'].select(range(64))

print('Look at one training example:')
display(train_ds[10])


Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Look at one training example:


{'tokens': ['Davidson',
  'College',
  '(',
  'Davidson',
  ',',
  'North',
  'Carolina',
  ')'],
 'ner_tags': [3, 4, 0, 5, 6, 6, 6, 0],
 'langs': ['en', 'en', 'en', 'en', 'en', 'en', 'en', 'en'],
 'spans': ['ORG: Davidson College', 'LOC: Davidson , North Carolina']}

In [6]:

def tokenize_and_align_labels(batch):
    tokenized = tokenizer_ft(batch['tokens'], is_split_into_words=True, truncation=True)
    aligned_labels = []
    for batch_index, labels in enumerate(batch['ner_tags']):
        word_ids = tokenized.word_ids(batch_index=batch_index)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                raw_label = tag_names[labels[word_idx]]
                if raw_label == 'O':
                    label_ids.append(label2id_ft['O'])
                else:
                    prefix, base_label = raw_label.split('-', 1)
                    mapped_label = f'{prefix}-{label_name_map[base_label]}'
                    label_ids.append(label2id_ft[mapped_label])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        aligned_labels.append(label_ids)
    tokenized['labels'] = aligned_labels
    return tokenized

tokenized_train_ds = train_ds.map(tokenize_and_align_labels, batched=True, remove_columns=['tokens', 'ner_tags', 'langs', 'spans'])
tokenized_eval_ds = eval_ds.map(tokenize_and_align_labels, batched=True, remove_columns=['tokens', 'ner_tags', 'langs', 'spans'])
data_collator = DataCollatorForTokenClassification(tokenizer_ft)

training_args = TrainingArguments(
    output_dir=os.path.join(CACHE_DIR, 'bert_ner_finetuned'),
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=10,
    save_strategy='no',
    eval_strategy='no',
    report_to='none',
)

DO_TRAIN = True
if DO_TRAIN:
    trainer = Trainer(
        model=model_ft,
        args=training_args,
        train_dataset=tokenized_train_ds,
        eval_dataset=tokenized_eval_ds,
        data_collator=data_collator,
    )
    trainer.train()
    metrics = trainer.evaluate()
    print(metrics)
    trainer.save_model()
else:
    print('Fine-tuning template ready: set DO_TRAIN = True to train on CoNLL-2003.')

Map:   0%|          | 0/64 [00:00<?, ? examples/s]

Step,Training Loss
10,1.783300


{'eval_loss': 1.564572811126709, 'eval_runtime': 0.0528, 'eval_samples_per_second': 1212.106, 'eval_steps_per_second': 75.757, 'epoch': 1.0}


In [7]:
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report

model_ft.eval()

true_labels = []
pred_labels = []
with torch.no_grad():
    for example in tokenized_eval_ds:
        labels = example['labels']
        inputs = {k: torch.tensor([v]).to(model_ft.device) for k, v in example.items() if k != 'labels'}
        logits = model_ft(**inputs).logits[0]
        pred_ids = logits.argmax(dim=-1).tolist()

        seq_true = []
        seq_pred = []
        for pred_id, label_id in zip(pred_ids, labels):
            if label_id == -100:
                continue
            seq_true.append(id2label_ft[int(label_id)])
            seq_pred.append(id2label_ft[int(pred_id)])
        true_labels.append(seq_true)
        pred_labels.append(seq_pred)

precision = precision_score(true_labels, pred_labels)
recall = recall_score(true_labels, pred_labels)
f1 = f1_score(true_labels, pred_labels)

metrics = {'precision': precision, 'recall': recall, 'f1': f1}
print('Validation precision:', precision)
print('Validation recall:', recall)
print('Validation f1:', f1)
print(classification_report(true_labels, pred_labels))

Validation precision: 0.0
Validation recall: 0.0
Validation f1: 0.0
              precision    recall  f1-score   support

    LOCATION       0.00      0.00      0.00        24
         ORG       0.00      0.00      0.00        30
      PERSON       0.00      0.00      0.00        28

   micro avg       0.00      0.00      0.00        82
   macro avg       0.00      0.00      0.00        82
weighted avg       0.00      0.00      0.00        82



/data01/pferrazzi/miniconda3/envs/crf/lib/python3.12/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


### Load models already trained for NER

In some cases, you do not want to do the training, but load a model already trained for NER instead:

In [8]:
BERT_NER_ID = 'dslim/bert-base-NER'
# Load tokenizer and token classification model directly (no pipeline)
tokenizer_bert_ner = AutoTokenizer.from_pretrained(BERT_NER_ID, cache_dir=CACHE_DIR)
model_bert_ner = AutoModelForTokenClassification.from_pretrained(BERT_NER_ID, cache_dir=CACHE_DIR)
print(model_bert_ner)
id2label = model_bert_ner.config.id2label
print('Trained to assign these labels: ', id2label)

tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12

Let's look at how to use the model once it has been loaded:

In [11]:
# First, identify the offset of each token in the original text
text = 'Barack Obama visited Stanford University in California'
enc_offsets = tokenizer_bert_ner(text, return_offsets_mapping=True, truncation=True)
offsets = enc_offsets['offset_mapping']
enc_offsets

{'input_ids': [101, 14319, 7661, 3891, 8036, 1239, 1107, 1756, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1], 'offset_mapping': [(0, 0), (0, 6), (7, 12), (13, 20), (21, 29), (30, 40), (41, 43), (44, 54), (0, 0)]}

In [12]:
# use the tokenizer and model to get the logits for the input text
# assign a probability to each token to belong to one of the classes
inputs = tokenizer_bert_ner(text, return_tensors='pt')
logits = model_bert_ner(**inputs).logits
print(logits.shape)
print('Logits for one token: ', logits[0][0])

# find the actual predictions based on the probabilities
pred_ids = logits.argmax(dim=-1)[0].tolist()
token_ids = inputs['input_ids'][0].tolist()


torch.Size([1, 9, 9])
Logits for one token:  tensor([ 8.0497, -0.4622, -1.0754, -0.3237, -1.4574, -1.0753, -1.9615, -1.2833,
        -1.1375], grad_fn=<SelectBackward0>)


In [13]:

def assign_labels_to_tokens(pred_ids, token_ids, text, verbose=False):
    entities = []
    current_label = None
    current_start = None
    current_end = None


    for i, (start, end) in enumerate(offsets):
        tok = tokenizer_bert_ner.convert_ids_to_tokens([token_ids[i]])[0]
        if tok == '[CLS]':
            continue
        if verbose: print(f"Token: {tok}")
        # Skip special tokens or tokens without character span
        if (start == 0 and end == 0):
            if verbose: print(f"  not an entity")
            pred_lbl = 'O'
        else:
            raw_lbl = id2label[pred_ids[i]]
            pred_lbl = 'O' if raw_lbl == 'O' else raw_lbl
            if verbose: print(f"  found label: {raw_lbl}")
        if pred_lbl != 'O':
            if current_label == pred_lbl and current_end == start:
                # extend current span
                current_end = end
                if verbose: print(f" this token is part of an already found entity!")
            else:
                # close any previous span
                if current_label is not None:
                    span_text = text[current_start:current_end]
                    entities.append({'text': span_text, 'label': current_label})
                    if verbose: print(f"  end of the entity")
                # start new span
                current_label = pred_lbl
                current_start = start
                current_end = end
        else:
            if current_label is not None:
                span_text = text[current_start:current_end]
                entities.append({'text': span_text, 'label': current_label})
                current_label = None
                current_start = None
                current_end = None
    # close tail span if any
    if current_label is not None:
        span_text = text[current_start:current_end]
        entities.append({'text': span_text, 'label': current_label})

    return entities

In [14]:
entities = assign_labels_to_tokens(pred_ids, token_ids, text, verbose=False)
display(entities)

[{'text': 'Barack', 'label': 'B-PER'},
 {'text': 'Obama', 'label': 'I-PER'},
 {'text': 'Stanford', 'label': 'B-ORG'},
 {'text': 'University', 'label': 'I-ORG'},
 {'text': 'California', 'label': 'B-LOC'}]

## Part 2: Gemma prompting for NER
We prompt a chat model to extract entities and return JSON with keys `PERSON`, `ORG`, `LOC`.
We demonstrate zero-shot and few-shot (3-shot) prompting.

### How We Evaluate Gemma on the Custom Dataset
We evaluate Gemma on the same small custom set used in the lab (`DATA`) with a simple pipeline:

1) Read the gold entities (`PERSON`, `ORG`, `LOC`) for each sentence.
2) Ask Gemma to return entities in JSON (zero-shot or one-shot).
3) Parse the model output robustly:
   - try direct JSON parsing,
   - if needed, extract the first balanced `{ ... }` JSON object,
   - if parsing still fails, use empty predictions for that sample.
4) Normalize entity strings (trim, collapse spaces, lowercase).
5) Convert gold and predicted entities to token-level BIO tags (`B-...`, `I-...`, `O`).
6) Compute precision, recall, and F1 with `seqeval.metrics` over all sentences.

The next cell implements these steps and prints metrics plus a classification report for both zero-shot and one-shot prompting.

In [15]:
import os, json, re
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification, AutoModelForCausalLM, pipeline

BASE_DIR = os.path.join('lab4/')
# BASE_DIR = os.path.join('../../lab4')
DATA_DIR = os.path.join(BASE_DIR, 'data')
CACHE_DIR = os.path.join(BASE_DIR, 'models_cache')
os.makedirs(CACHE_DIR, exist_ok=True)

print('Transformers:', __import__('transformers').__version__)
print('Torch:', torch.__version__)

Transformers: 4.56.0
Torch: 2.8.0+cu128


In [16]:
with open(os.path.join(DATA_DIR, 'ner_examples.json'), 'r', encoding='utf-8') as f:
    DATA = json.load(f)
len(DATA), DATA[0]['text']

(10, 'Barack Obama visited Miami Beach in Florida.')

In [17]:
from transformers import AutoTokenizer, AutoModelForCausalLM
GEMMA_ID = 'unsloth/gemma-3-1B-it'

def load_chat_model(model_id):
    tok = AutoTokenizer.from_pretrained(model_id, cache_dir=CACHE_DIR)
    mdl = AutoModelForCausalLM.from_pretrained(model_id, cache_dir=CACHE_DIR, torch_dtype=torch.float16, device_map='auto')
    return tok, mdl

tokenizer_chat, model_chat = load_chat_model(GEMMA_ID)

print('Chat model:', GEMMA_ID)
print('Has chat template?', bool(getattr(tokenizer_chat, 'chat_template', None)))


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/902 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

Chat model: unsloth/gemma-3-1B-it
Has chat template? True


In [18]:

SYSTEM_PROMPT = (
    'You extract named entities from text. Return a JSON object with keys PERSON,'
    'ORG, LOC, each mapped to an array of strings.'
    'Use exact surface forms from the text and avoid duplicates.'
)

def build_messages_zero_shot(text):
    return [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': 'Text:' + text + 'Return only JSON.'}
    ]


def build_messages_with_shot(text, examples):
    msgs = [{'role': 'system', 'content': SYSTEM_PROMPT}]
    for ex in examples:
        msgs.append({'role': 'user', 'content': ex['input']})
        msgs.append({'role': 'assistant', 'content': f"Here start the examples:\n{json.dumps(ex['output'])}"})
    msgs.append({'role': 'user', 'content': f'HERE IS THE INPUT TEXT: {text}'})
    return msgs

def generate_json_entities(text, few_shot=False, examples=None, max_new_tokens=128, verbose=False):
    messages = build_messages_with_shot(text, examples) if few_shot else build_messages_zero_shot(text)
    processed_inputs = tokenizer_chat.apply_chat_template(messages, return_tensors='pt')
    if verbose:
        print('INPUT GIVEN TO THE MODEL:')
        print(tokenizer_chat.decode(processed_inputs[0], skip_special_tokens=False), '\n---\n')
    input_ids = processed_inputs.to(model_chat.device)
    gen = model_chat.generate(input_ids, max_new_tokens=max_new_tokens, temperature=0.01)
    output = gen[0][input_ids.shape[1]:]  # only the generated part
    out = tokenizer_chat.decode(output, skip_special_tokens=False)
    return out

# Generation examples
# Few-shot examples
with open(os.path.join(DATA_DIR, 'few_shot_ner_examples.json'), 'r', encoding='utf-8') as f:
    FEW = json.load(f)

print('ZERO-SHOT output:', generate_json_entities(DATA[0]['text'], few_shot=False, verbose=True))
print("==========================\n==========================\n")
print('ONE-SHOT output:', generate_json_entities(DATA[0]['text'], few_shot=True, examples=FEW[:1], verbose=True))

INPUT GIVEN TO THE MODEL:
<bos><start_of_turn>user
You extract named entities from text. Return a JSON object with keys PERSON,ORG, LOC, each mapped to an array of strings.Use exact surface forms from the text and avoid duplicates.

Text:Barack Obama visited Miami Beach in Florida.Return only JSON.<end_of_turn>
 
---

ZERO-SHOT output: ```json
{
  "PERSON": ["Barack Obama"],
  "ORG": ["Miami Beach"],
  "LOC": ["Florida"]
}
```
<end_of_turn>

INPUT GIVEN TO THE MODEL:
<bos><start_of_turn>user
You extract named entities from text. Return a JSON object with keys PERSON,ORG, LOC, each mapped to an array of strings.Use exact surface forms from the text and avoid duplicates.

Barack Obama spoke at Stanford University in California.<end_of_turn>
<start_of_turn>model
Here start the examples:
{"PERSON": ["Barack Obama"], "ORG": ["Stanford University"], "LOC": ["California"]}<end_of_turn>
<start_of_turn>user
HERE IS THE INPUT TEXT: Barack Obama visited Miami Beach in Florida.<end_of_turn>
 
---


In [19]:
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report

def parse_ner_json(raw):
    empty = {k: set() for k in ('PERSON', 'ORG', 'LOC')}
    if raw is None:
        return empty

    text = str(raw).strip()
    text = text.replace('```json', '').replace('```', '').strip()

    # Try direct JSON first.
    try:
        obj = json.loads(text)
    except Exception:
        # Fallback: extract first balanced JSON object from the text.
        start = text.find('{')
        if start < 0:
            return empty

        depth = 0
        end = None
        for i in range(start, len(text)):
            ch = text[i]
            if ch == '{':
                depth += 1
            elif ch == '}':
                depth -= 1
                if depth == 0:
                    end = i + 1
                    break

        if end is None:
            return empty

        candidate = text[start:end]
        try:
            obj = json.loads(candidate)
        except Exception:
            return empty

    out = {k: set() for k in ('PERSON', 'ORG', 'LOC')}
    for k in out:
        vals = obj.get(k, []) if isinstance(obj, dict) else []
        if isinstance(vals, list):
            out[k] = {re.sub(r'\s+', ' ', str(x).strip()).lower() for x in vals}
    return out

def gold_ner_sets(entry):
    out = {k: set() for k in ('PERSON', 'ORG', 'LOC')}
    for ent in entry.get('entities', []):
        lbl = ent.get('label', '').upper()
        if lbl in out:
            out[lbl].add(re.sub(r'\s+', ' ', ent.get('text', '').strip()).lower())
    return out

def tokens_and_spans(text):
    matches = list(re.finditer(r'\S+', text))
    tokens = [m.group(0) for m in matches]
    spans = [(m.start(), m.end()) for m in matches]
    return tokens, spans

def entities_to_bio(text, entities_by_label):
    _, spans = tokens_and_spans(text)
    tags = ['O'] * len(spans)
    lower_text = text.lower()

    for label in ('PERSON', 'ORG', 'LOC'):
        for mention in sorted(entities_by_label.get(label, set()), key=len, reverse=True):
            if not mention:
                continue
            start = lower_text.find(mention)
            if start < 0:
                continue
            end = start + len(mention)
            idxs = [i for i, (s, e) in enumerate(spans) if not (e <= start or s >= end)]
            if not idxs:
                continue
            tags[idxs[0]] = f'B-{label}'
            for j in idxs[1:]:
                tags[j] = f'I-{label}'

    return tags

def evaluate_gemma_seqeval(few_shot=False, examples=None):
    y_true = []
    y_pred = []

    for row in DATA:
        text = row['text']
        gold = gold_ner_sets(row)
        raw = generate_json_entities(text, few_shot=few_shot, examples=examples)
        pred = parse_ner_json(raw)

        y_true.append(entities_to_bio(text, gold))
        y_pred.append(entities_to_bio(text, pred))

    metrics = {
        'precision': float(precision_score(y_true, y_pred)),
        'recall': float(recall_score(y_true, y_pred)),
        'f1': float(f1_score(y_true, y_pred)),
    }
    return metrics, y_true, y_pred

zero_metrics, y_true_zero, y_pred_zero = evaluate_gemma_seqeval(few_shot=False)
one_metrics, y_true_one, y_pred_one = evaluate_gemma_seqeval(few_shot=True, examples=FEW[:1])

print('Gemma zero-shot (seqeval):', zero_metrics)
print(classification_report(y_true_zero, y_pred_zero))

print('Gemma one-shot (seqeval):', one_metrics)
print(classification_report(y_true_one, y_pred_one))

Gemma zero-shot (seqeval): {'precision': 0.7, 'recall': 0.7241379310344828, 'f1': 0.711864406779661}
              precision    recall  f1-score   support

         LOC       0.90      0.82      0.86        11
         ORG       0.60      0.55      0.57        11
      PERSON       0.60      0.86      0.71         7

   micro avg       0.70      0.72      0.71        29
   macro avg       0.70      0.74      0.71        29
weighted avg       0.71      0.72      0.71        29

Gemma one-shot (seqeval): {'precision': 0.8571428571428571, 'recall': 0.6206896551724138, 'f1': 0.7200000000000001}
              precision    recall  f1-score   support

         LOC       1.00      0.64      0.78        11
         ORG       0.67      0.55      0.60        11
      PERSON       1.00      0.71      0.83         7

   micro avg       0.86      0.62      0.72        29
   macro avg       0.89      0.63      0.74        29
weighted avg       0.87      0.62      0.72        29



### Excercise

Build a prediction script on the first 64 examples of `wikiann` (`en`) using:
- BERT NER model
- Llama 1B chat model (for entity extraction via prompt)

The steps to follow are:
1. load the data
2. load the BERT model
3. train the BERT model
4. evaluate its performances using the same functions of this notebook
5. find the best prompting strategy for gemma
6. construct an accurate function to parse the output of gemma and be able to evaluate it (the approach in this notebbok might be suboptimal!)
7. evaluate its performances

In [ ]:
from datasets import load_dataset

wikiann = load_dataset('wikiann', 'en')
wikiann['train'] = wikiann['train'].select(range(N_TRAIN_EXAMPLES)) 
label_list = wikiann['train'].features['ner_tags'].feature.names
label_to_id = {label: i for i, label in enumerate(label_list)}
id_to_label = {i: label for label, i in label_to_id.items()}